# Step 6 Imaging-only DDT From FPD

Read the imaging-only posterior with precomputed FPD values, build a joint 1D grid posterior for a single `D_dt` per sample, and inspect the 6 chains.

In [2]:
import os
import sys
from pathlib import Path
from matplotlib.lines import Line2D

os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('JAX_PLATFORM_NAME', 'cpu')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')

import numpy as np
import arviz as az
import xarray as xr
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

WFI2033_DIR = Path('/mnt/d/lensing/Herculens/Herculensedquasar/WFI2033')
if str(WFI2033_DIR) not in sys.path:
    sys.path.insert(0, str(WFI2033_DIR))

from Tian_infra import import_function
import_function(globals())

jax.config.update('jax_enable_x64', True)
numpyro.enable_x64()

INPUT_PATH = Path('/mnt/lustre/tianli/quasar_hmc/WFI2033_ss=2_inferh0_step6_imaging_only_20260413_19/WFI2033_all_ss=2_inferh0_step6_imaging_only_withFPD.nc')
INPUT_PATH


PosixPath('/mnt/lustre/tianli/quasar_hmc/WFI2033_ss=2_inferh0_step6_imaging_only_20260413_19/WFI2033_all_ss=2_inferh0_step6_imaging_only_withFPD.nc')

In [2]:
Z_LENS = 0.6575
Z_SOURCE = 1.662
TIME_DELAY_OBS = {
    'dt_31_days': {'mean': -36.2, 'sigma_minus': 2.3, 'sigma_plus': 1.6},
    'dt_32_days': {'mean': -37.3, 'sigma_minus': 3.0, 'sigma_plus': 2.6},
    'dt_34_days': {'mean': -59.4, 'sigma_minus': 1.3, 'sigma_plus': 1.3},
}

ARCSEC_TO_RAD = np.deg2rad(1.0 / 3600.0)
MPC_TO_KM = 3.0856775814913673e19
DAY_TO_S = 86400.0
TIME_DELAY_SCALE_DAYS = MPC_TO_KM * ARCSEC_TO_RAD**2 / (Cosmo.c_km_s * DAY_TO_S)

COSMO_PRIOR_MODE = 'uniform'  # 'uniform' or 'gaussian'
COSMO_PRIOR_NAME = 'DESI_PLANCK'
OMEGA_M_LOW = 0.05
OMEGA_M_HIGH = 0.6
H0_LOW = 60.0
H0_HIGH = 80.0
COSMO_NUM_WARMUP = 1000
COSMO_NUM_SAMPLES = 1000
COSMO_TARGET_ACCEPT = 0.9
COSMO_MAX_TREE_DEPTH = 10
CORNER_THIN = 2
FPD_COV_REG_FRAC = 1e-6
FPD_COV_REG_FLOOR = 1e-10
H0_BLIND_SEED = 20331
H0_BLIND_OFFSET = float(np.random.default_rng(H0_BLIND_SEED).uniform(4.0, 9.0))

DESI_PLANCK_cov = np.array([
    [1.3204061745961228e-05, -0.000999369023951933],
    [-0.000999369023951933, 0.07954650644002863],
], dtype=float)
PantheonSH0ES_cov = np.array([
    [0.0003513006147410213, -0.0026497789652042587],
    [-0.0026497789652042587, 1.1406699901973247],
], dtype=float)

COSMO_PRIORS = {
    'DESI_PLANCK': {
        'mean_vec': np.array([0.302677, 68.171897], dtype=float),
        'cov': DESI_PLANCK_cov,
    },
    'PantheonSH0ES': {
        'mean_vec': np.array([0.332865, 73.502685], dtype=float),
        'cov': PantheonSH0ES_cov,
    },
}

FPD_LABELS = [r'$\Delta\phi_{31}$', r'$\Delta\phi_{32}$', r'$\Delta\phi_{34}$']
COSMO_LABELS = [r'$\Omega_m$', r'$H_0$ blinded']
CHAIN_COLORS = plt.cm.tab10(np.arange(6))


def regularize_covariance(cov, frac=FPD_COV_REG_FRAC, floor=FPD_COV_REG_FLOOR):
    cov = np.asarray(cov, dtype=float)
    reg = max(floor, frac * np.trace(cov) / cov.shape[0])
    return cov + np.eye(cov.shape[0], dtype=float) * reg


def build_cosmology(omega_m, h0):
    return {
        'Omegam': jnp.asarray(omega_m, dtype=jnp.float64),
        'Omegak': jnp.asarray(0.0, dtype=jnp.float64),
        'w0': jnp.asarray(-1.0, dtype=jnp.float64),
        'wa': jnp.asarray(0.0, dtype=jnp.float64),
        'h0': jnp.asarray(h0, dtype=jnp.float64),
    }


def compute_d_dt_mpc(omega_m, h0):
    return np.asarray(
        jnp.squeeze(Cosmo.compute_time_delay_distances(build_cosmology(omega_m, h0), Z_LENS, Z_SOURCE)),
        dtype=float,
    ).reshape(-1)


def estimate_chain_fpd_stats(posterior, chain_idx):
    raw = np.column_stack([
        np.asarray(posterior['fpd_31'].isel(chain=chain_idx).values, dtype=float),
        np.asarray(posterior['fpd_32'].isel(chain=chain_idx).values, dtype=float),
        np.asarray(posterior['fpd_34'].isel(chain=chain_idx).values, dtype=float),
    ])
    mean = raw.mean(axis=0)
    cov = regularize_covariance(np.cov(raw, rowvar=False, ddof=1))
    return raw, mean, cov


def sample_cosmology_prior():
    if COSMO_PRIOR_MODE == 'uniform':
        omega_m = numpyro.sample('omega_m', dist.Uniform(OMEGA_M_LOW, OMEGA_M_HIGH))
        h0_true = numpyro.sample('h0_raw', dist.Uniform(H0_LOW, H0_HIGH))
        return omega_m, h0_true

    if COSMO_PRIOR_MODE == 'gaussian':
        prior = COSMO_PRIORS[COSMO_PRIOR_NAME]
        cosmo_vec = numpyro.sample(
            'cosmo_vec',
            dist.MultivariateNormal(
                loc=jnp.asarray(prior['mean_vec'], dtype=jnp.float64),
                covariance_matrix=jnp.asarray(prior['cov'], dtype=jnp.float64),
            ),
        )
        return cosmo_vec[0], cosmo_vec[1]

    raise ValueError(f'Unsupported COSMO_PRIOR_MODE: {COSMO_PRIOR_MODE}')


def cosmology_from_fpd_model(fpd_mean, fpd_cov):
    fpd_vec = numpyro.sample(
        'fpd_vec',
        dist.MultivariateNormal(
            loc=jnp.asarray(fpd_mean, dtype=jnp.float64),
            covariance_matrix=jnp.asarray(fpd_cov, dtype=jnp.float64),
        ),
    )
    omega_m, h0_true = sample_cosmology_prior()
    d_dt = jnp.squeeze(Cosmo.compute_time_delay_distances(build_cosmology(omega_m, h0_true), Z_LENS, Z_SOURCE))
    prefactor_days = d_dt * jnp.asarray(TIME_DELAY_SCALE_DAYS, dtype=jnp.float64)

    dt_31 = prefactor_days * fpd_vec[0]
    dt_32 = prefactor_days * fpd_vec[1]
    dt_34 = prefactor_days * fpd_vec[2]

    numpyro.factor(
        'dt_31_like',
        Numpyro_function.split_normal_logpdf(
            dt_31,
            TIME_DELAY_OBS['dt_31_days']['mean'],
            TIME_DELAY_OBS['dt_31_days']['sigma_minus'],
            TIME_DELAY_OBS['dt_31_days']['sigma_plus'],
        ),
    )
    numpyro.factor(
        'dt_32_like',
        Numpyro_function.split_normal_logpdf(
            dt_32,
            TIME_DELAY_OBS['dt_32_days']['mean'],
            TIME_DELAY_OBS['dt_32_days']['sigma_minus'],
            TIME_DELAY_OBS['dt_32_days']['sigma_plus'],
        ),
    )
    numpyro.factor(
        'dt_34_like',
        Numpyro_function.split_normal_logpdf(
            dt_34,
            TIME_DELAY_OBS['dt_34_days']['mean'],
            TIME_DELAY_OBS['dt_34_days']['sigma_minus'],
            TIME_DELAY_OBS['dt_34_days']['sigma_plus'],
        ),
    )


def run_chain_cosmology_inference(fpd_mean, fpd_cov, rng_seed):
    kernel = NUTS(
        cosmology_from_fpd_model,
        target_accept_prob=COSMO_TARGET_ACCEPT,
        max_tree_depth=COSMO_MAX_TREE_DEPTH,
    )
    mcmc = MCMC(
        kernel,
        num_warmup=COSMO_NUM_WARMUP,
        num_samples=COSMO_NUM_SAMPLES,
        num_chains=1,
        progress_bar=True,
    )
    mcmc.run(jax.random.PRNGKey(rng_seed), fpd_mean=jnp.asarray(fpd_mean), fpd_cov=jnp.asarray(fpd_cov))
    samples = {k: np.asarray(v) for k, v in mcmc.get_samples().items()}

    if COSMO_PRIOR_MODE == 'uniform':
        omega_m = np.asarray(samples['omega_m'], dtype=float).reshape(-1)
        h0_raw = np.asarray(samples['h0_raw'], dtype=float).reshape(-1)
    else:
        cosmo_vec = np.asarray(samples['cosmo_vec'], dtype=float)
        omega_m = cosmo_vec[:, 0]
        h0_raw = cosmo_vec[:, 1]

    fpd_vec = np.asarray(samples['fpd_vec'], dtype=float)
    d_dt = compute_d_dt_mpc(omega_m, h0_raw)
    h0_blinded = h0_raw + H0_BLIND_OFFSET

    return {
        'fpd_vec': fpd_vec,
        'OmegaM': omega_m,
        'H0_blinded': h0_blinded,
        'D_dt_Mpc': d_dt,
    }


def overlay_corner(samples_by_chain, labels, title):
    fig = None
    for chain_idx, samples in enumerate(samples_by_chain):
        fig = corner.corner(
            samples[::CORNER_THIN],
            fig=fig,
            labels=labels,
            color=CHAIN_COLORS[chain_idx],
            plot_datapoints=False,
            fill_contours=False,
            no_fill_contours=True,
            smooth=1.0,
            hist_bin_factor=1.5,
            levels=(0.68, 0.95),
            label_kwargs={'fontsize': 13},
            contour_kwargs={'linewidths': 1.6},
            hist_kwargs={'density': True, 'linewidth': 1.6},
            show_titles=False,
        )
    handles = [Line2D([0], [0], color=CHAIN_COLORS[i], lw=2, label=f'chain {i}') for i in range(len(samples_by_chain))]
    fig.legend(handles=handles, loc='upper right', frameon=False)
    fig.suptitle(title, y=0.98)
    plt.show()
    return fig


In [4]:
posterior = xr.open_dataset(
    INPUT_PATH,
    group='posterior',
    engine='h5netcdf',
)[['fpd_31', 'fpd_32', 'fpd_34']].load()
n_chain = posterior.sizes['chain']
n_draw = posterior.sizes['draw']

raw_fpd_by_chain = []
fpd_mean = np.empty((n_chain, 3), dtype=float)
fpd_cov = np.empty((n_chain, 3, 3), dtype=float)

for c in range(n_chain):
    raw, mean, cov = estimate_chain_fpd_stats(posterior, c)
    raw_fpd_by_chain.append(raw)
    fpd_mean[c, :] = mean
    fpd_cov[c, :, :] = cov

fpd_stats = xr.Dataset(
    data_vars={
        'fpd_mean': (('chain', 'fpd_component'), fpd_mean),
        'fpd_cov': (('chain', 'fpd_component', 'fpd_component_2'), fpd_cov),
    },
    coords={
        'chain': posterior.coords['chain'].values,
        'fpd_component': ['fpd_31', 'fpd_32', 'fpd_34'],
        'fpd_component_2': ['fpd_31', 'fpd_32', 'fpd_34'],
    },
)
fpd_stats


<xarray.Dataset> Size: 768B
Dimensions:          (chain: 6, fpd_component: 3, fpd_component_2: 3)
Coordinates:
  * chain            (chain) int64 48B 0 1 2 3 4 5
  * fpd_component    (fpd_component) <U6 72B 'fpd_31' 'fpd_32' 'fpd_34'
  * fpd_component_2  (fpd_component_2) <U6 72B 'fpd_31' 'fpd_32' 'fpd_34'
Data variables:
    fpd_mean         (chain, fpd_component) float64 144B nan nan nan ... nan nan
    fpd_cov          (chain, fpd_component, fpd_component_2) float64 432B nan...

In [6]:
chain_results = []
for c in range(n_chain):
    print(f'Running cosmology inference for chain {c} ...')
    chain_results.append(run_chain_cosmology_inference(fpd_mean[c], fpd_cov[c], rng_seed=9100 + c))

n_post = len(chain_results[0]['OmegaM'])
posterior_cosmo = xr.Dataset(
    data_vars={
        'fpd_31_latent': (('chain', 'sample'), np.stack([r['fpd_vec'][:, 0] for r in chain_results], axis=0)),
        'fpd_32_latent': (('chain', 'sample'), np.stack([r['fpd_vec'][:, 1] for r in chain_results], axis=0)),
        'fpd_34_latent': (('chain', 'sample'), np.stack([r['fpd_vec'][:, 2] for r in chain_results], axis=0)),
        'OmegaM': (('chain', 'sample'), np.stack([r['OmegaM'] for r in chain_results], axis=0)),
        'H0_blinded': (('chain', 'sample'), np.stack([r['H0_blinded'] for r in chain_results], axis=0)),
        'D_dt_Mpc': (('chain', 'sample'), np.stack([r['D_dt_Mpc'] for r in chain_results], axis=0)),
    },
    coords={
        'chain': posterior.coords['chain'].values,
        'sample': np.arange(n_post),
    },
    attrs={
        'cosmo_prior_mode': COSMO_PRIOR_MODE,
        'cosmo_prior_name': COSMO_PRIOR_NAME,
        'h0_blinded_note': 'A constant blind offset has been added to H0 before plotting and storage.',
    },
)
posterior_cosmo


Running cosmology inference for chain 0 ...


ValueError: MultivariateNormal distribution got invalid loc parameter.

In [ ]:
fpd_corner_samples = [
    np.column_stack([
        posterior_cosmo['fpd_31_latent'].isel(chain=c).values,
        posterior_cosmo['fpd_32_latent'].isel(chain=c).values,
        posterior_cosmo['fpd_34_latent'].isel(chain=c).values,
    ])
    for c in range(n_chain)
]

overlay_corner(
    fpd_corner_samples,
    FPD_LABELS,
    'Latent FPD posterior by chain',
)

cosmo_corner_samples = [
    np.column_stack([
        posterior_cosmo['OmegaM'].isel(chain=c).values,
        posterior_cosmo['H0_blinded'].isel(chain=c).values,
    ])
    for c in range(n_chain)
]

overlay_corner(
    cosmo_corner_samples,
    COSMO_LABELS,
    'Cosmology posterior by chain (H0 is blinded by a constant offset)',
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for c in range(n_chain):
    axes[0].plot(posterior_cosmo['sample'], posterior_cosmo['OmegaM'].isel(chain=c), color=CHAIN_COLORS[c], alpha=0.8, label=f'chain {c}')
    axes[1].plot(posterior_cosmo['sample'], posterior_cosmo['H0_blinded'].isel(chain=c), color=CHAIN_COLORS[c], alpha=0.8)

axes[0].set_ylabel(r'$\Omega_m$')
axes[1].set_ylabel(r'$H_0$ blinded')
axes[1].set_xlabel('sample')
axes[0].legend(ncol=3, fontsize=9)
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
